In [66]:
#Import libraries
import pandas as pd
import numpy as np

In [67]:
#Download Data
data = pd.read_excel("../data/ctech projects (CA).xlsx")

print("Shape:", data.shape)

Shape: (46608, 30)


In [68]:
#Create Modeling Dataset
model_data = data.copy()

In [69]:
#Apply ENG business rules

model_data["Has Test Task Flag"].value_counts()

Has Test Task Flag
No     25560
Yes    21048
Name: count, dtype: int64

In [70]:
#Filter to create a ENG focused datset only. Filter to only keep No values

model_data = model_data[
    model_data["Has Test Task Flag"] == "No"
]

print(model_data.shape)

(25560, 30)


In [ ]:
#The Stats of this specific columnn
model_data["Actual Eng Hours"].describe()


count    25044.000000
mean        13.651743
std         29.461530
min          0.000000
25%          4.000000
50%          8.000000
75%         15.700000
max       3000.000000
Name: Avg Scoped Hours Engineer, dtype: float64

In [ ]:
#Total rows with missing hours
print(
    model_data["Actual Eng Hours"].isna().sum()
)

516


In [73]:
#How many missing values does each column still have?
model_data.isna().sum().sort_values(ascending=False)

Product Type Segment            24835
Avg Scoped Hours Lab            17615
Flex Standards                  17230
Product Group                    7616
CCN                              4240
Avg Scoped Hours Engineer         516
Actual Lab Hours                  250
Actual Eng Hours                  250
Service Program                   221
Service Detail                    214
Product Type                       72
Handler Region                      0
Product Type Code                   0
Confirmation Year                   0
Service Catalog Category            0
Service Line Name                   0
Industry Code                       0
Service Line Code                   0
Industry Name                       0
Flex Project UUID                   0
Service Catalog Segment             0
Service Catalog Item Number         0
Ship to Account Number              0
Has Test Task Flag                  0
Ship to Customer Region             0
Service Catalog Sub Category        0
Flex Project

In [ ]:
#1. Drop rows where the ENG target variable is missing
model_data = model_data.dropna( subset =["Actual Eng Hours"])

In [ ]:
#2. Create log-transformed target
model_data["Log_Eng_Hours"] = np.log1p(model_data["Actual Eng Hours"])

In [76]:
#Drop obvious unneccessary /leakage columns
drop_cols = [
    "Flex Project UUID",
    "Actual Eng Hours",
    "Actual Lab Hours",
    "Total Scoped Hours",
    "Product Type Segment",
    "Avg Scoped Hours Lab"
]

model_data = model_data.drop(columns=drop_cols, errors="ignore")

In [77]:
cat_fill_cols = [
    "Flex Standards",
    "Product Group",
    "CCN",
    "Service Program",
    "Service Detail",
    "Product Type"
]

for col in cat_fill_cols:
    model_data[col] = model_data[col].fillna("UNKNOWN")

In [ ]:
#Create the log target
model_data["Log_Eng_Hours"] = np.log1p(
    model_data["Actual Eng Hours"]
)

In [ ]:
model_data = model_data.dropna(
    subset=["Actual Eng Hours"]
)

In [80]:
#Double check missing values again
model_data.isna().sum().sort_values(ascending=False)

Confirmation Year               0
Handler Region                  0
Product Group                   0
Product Type                    0
Product Type Code               0
Industry Name                   0
Industry Code                   0
Service Line Code               0
Service Line Name               0
Service Detail                  0
Service Program                 0
Service Catalog Category        0
Service Catalog Item Number     0
Service Catalog Segment         0
Service Catalog Sub Category    0
CCN                             0
Ship to Customer Region         0
Has Test Task Flag              0
Ship to Account Number          0
Flex Standards                  0
Standard Count                  0
Avg Scoped Hours Engineer       0
Flex Project Count              0
Test Count                      0
Log_Eng_Hours                   0
dtype: int64

In [ ]:
#List all the columns we have now after cleaning
model_data.columns.tolist()

['Confirmation Year',
 'Handler Region',
 'Product Group',
 'Product Type',
 'Product Type Code',
 'Industry Name',
 'Industry Code',
 'Service Line Code',
 'Service Line Name',
 'Service Detail',
 'Service Program',
 'Service Catalog Category',
 'Service Catalog Item Number',
 'Service Catalog Segment',
 'Service Catalog Sub Category',
 'CCN',
 'Ship to Customer Region',
 'Has Test Task Flag',
 'Ship to Account Number',
 'Flex Standards',
 'Standard Count',
 'Avg Scoped Hours Engineer',
 'Flex Project Count',
 'Test Count',
 'Log_Eng_Hours']

In [ ]:
#This is the cleaned modeling dataset we will use for machine learning

# Features (X)
X = model_data.drop(
    columns=[
        "Actual Eng Hours",
        "Log_Eng_Hours"
    ]
)

# Target (y)
y = model_data["Log_Eng_Hours"]

print(X.shape)
print(y.shape)


(25044, 23)
(25044,)


In [ ]:
#find all text/categorical columns that need to be converted into numeric ML features
cat_cols = X.select_dtypes(include=["object"]).columns

print(cat_cols)
print("Total categorical columns:", len(cat_cols))


Index(['Confirmation Year', 'Handler Region', 'Product Group', 'Product Type',
       'Product Type Code', 'Industry Name', 'Industry Code',
       'Service Line Code', 'Service Line Name', 'Service Detail',
       'Service Program', 'Service Catalog Category',
       'Service Catalog Item Number', 'Service Catalog Segment',
       'Service Catalog Sub Category', 'CCN', 'Ship to Customer Region',
       'Has Test Task Flag', 'Flex Standards'],
      dtype='str')
Total categorical columns: 19


C:\Users\105802\AppData\Local\Temp\ipykernel_23864\3974864412.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns


In [ ]:
#Convert all categorical/text columns into numeric columns that the ML model can understand
X_encoded = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True
)

print(X_encoded.shape)


(25044, 5297)


In [ ]:
#Double check that all columns are now numeric data types for machine learning
X_encoded.dtypes.value_counts()

bool     5293
int64       4
Name: count, dtype: int64

In [ ]:
#Verify that no text/oject columns are left after encoding
X_encoded.select_dtypes(include=["object"]).columns

Index([], dtype='str')

In [90]:
###################################MODEL TRAINING STARTS HERE ############################################
# Clean column names for XGBoost
X_encoded.columns = (
    X_encoded.columns
    .str.replace("[", "_", regex=False)
    .str.replace("]", "_", regex=False)
    .str.replace("<", "_", regex=False)
)

In [91]:
# ==============================
# MODEL COMPARISON: 5 BASELINE MODELS
# ==============================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

# 2. Evaluation function
def evaluate_model(model, X_test, y_test):
    pred_log = model.predict(X_test)

    # Convert log predictions back to real engineering hours
    y_test_hours = np.expm1(y_test)
    pred_hours = np.expm1(pred_log)

    mae = mean_absolute_error(y_test_hours, pred_hours)
    rmse = np.sqrt(mean_squared_error(y_test_hours, pred_hours))
    r2 = r2_score(y_test, pred_log)

    return mae, rmse, r2


# ==============================
# MODEL 1: LINEAR REGRESSION
# ==============================

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_mae, linear_rmse, linear_r2 = evaluate_model(
    linear_model,
    X_test,
    y_test
)


# ==============================
# MODEL 2: RIDGE REGRESSION
# ==============================

ridge_model = Ridge()
ridge_model.fit(X_train, y_train)

ridge_mae, ridge_rmse, ridge_r2 = evaluate_model(
    ridge_model,
    X_test,
    y_test
)


# ==============================
# MODEL 3: LASSO REGRESSION
# ==============================

lasso_model = Lasso(max_iter=10000)
lasso_model.fit(X_train, y_train)

lasso_mae, lasso_rmse, lasso_r2 = evaluate_model(
    lasso_model,
    X_test,
    y_test
)


# ==============================
# MODEL 4: RANDOM FOREST
# ==============================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_mae, rf_rmse, rf_r2 = evaluate_model(
    rf_model,
    X_test,
    y_test
)


# ==============================
# MODEL 5: XGBOOST
# ==============================


xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_mae, xgb_rmse, xgb_r2 = evaluate_model(
    xgb_model,
    X_test,
    y_test
)


# ==============================
# COMPARE ALL MODEL RESULTS
# ==============================

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "Random Forest",
        "XGBoost"
    ],
    "MAE_hours": [
        linear_mae,
        ridge_mae,
        lasso_mae,
        rf_mae,
        xgb_mae
    ],
    "RMSE_hours": [
        linear_rmse,
        ridge_rmse,
        lasso_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R2_log_scale": [
        linear_r2,
        ridge_r2,
        lasso_r2,
        rf_r2,
        xgb_r2
    ]
})

results.sort_values("RMSE_hours")



c:\Users\105802\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.96523365992878e-18.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,Model,MAE_hours,RMSE_hours,R2_log_scale
3,Random Forest,5.165511,16.084828,0.635372
1,Ridge Regression,6.358706,17.806513,0.529438
4,XGBoost,6.460559,18.081431,0.529361
0,Linear Regression,6.669580,18.901273,0.489927
2,Lasso Regression,8.995614,22.160476,0.019134
